<a href="https://colab.research.google.com/github/yweslakarep123/Krispy2/blob/main/final_project_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

  import subprocess, os

  # 1. EGL config so libglvnd can find the Nvidia EGL driver
  NVIDIA_ICD = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
  os.makedirs(os.path.dirname(NVIDIA_ICD), exist_ok=True)
  if not os.path.exists(NVIDIA_ICD):
      with open(NVIDIA_ICD, 'w') as f:
          f.write('{\n  "file_format_version":"1.0.0",\n'
                  '  "ICD":{"library_path":"libEGL_nvidia.so.0"}\n}\n')

  # 2. Set render backend BEFORE any mujoco / dm_control import
  os.environ['MUJOCO_GL']         = 'egl'
  os.environ['PYOPENGL_PLATFORM'] = 'egl'

  # 3. Install missing packages (h5py, gymnasium-robotics, minari)
  #    dm_control, torch, numpy, tqdm are already in Colab.
  subprocess.run([
      'pip', 'install', '-q',
      'dm_control>=1.0.38',          # MuJoCo Python bindings
      'gymnasium-robotics>=1.2.4',   # FrankaKitchen-v1 env
      'h5py',                        # read .hdf5 dataset files
      'minari',                      # fallback dataset source (d4rl successor)
  ], check=True)

  print("Setup complete.")

Setup complete.


In [ ]:
"""
FlowPolicy + FQL Refinement + RST Data Augmentation
for Franka Kitchen (kitchen-complete-v2 via Minari, or HDF5 fallback).

Compatible with Google Colab (Python 3.12, GPU runtime).
Uses Minari (D4RL/kitchen/complete-v2) by default; no d4rl package required.

────────────────────────────────────────────────────────────────
HOW TO RUN ON COLAB
────────────────────────────────────────────────────────────────
Paste and run SECTION 0 in its own cell first, then run the rest.
"""

# =============================================================================
# SECTION 0  --  COLAB SETUP  (run this cell alone, then continue)
# =============================================================================
# +-------------------------------------------------------------------------+
# |  Copy everything inside this block into a Colab cell and run it.        |
# |  A kernel restart is NOT needed after installation.                     |
# +-------------------------------------------------------------------------+
#


# =============================================================================
# SECTION 1  --  IMPORTS
# =============================================================================
import os, sys, math, copy, random, warnings, urllib.request
warnings.filterwarnings("ignore")

# Must be set before any MuJoCo / dm_control import
os.environ.setdefault("MUJOCO_GL",         "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from typing import Dict, List, Tuple, Optional
from tqdm import tqdm
import imageio

# MuJoCo via dm_control  (direct `import mujoco` crashes on Colab)
try:
    from dm_control import mujoco as _dm_mujoco   # noqa: F401
    print("dm_control loaded")
except ImportError:
    print("WARNING: dm_control not found -- run Section 0 first")

# gymnasium-robotics  (registers FrankaKitchen-v1)
try:
    import gymnasium as gym
    import gymnasium_robotics                      # noqa: F401
    gym.register_envs(gymnasium_robotics)
    print(f"gymnasium {gym.__version__} + gymnasium-robotics loaded")
except ImportError:
    print("WARNING: gymnasium-robotics not found -- run Section 0 first")

try:
    import h5py
    print("h5py loaded")
except ImportError:
    print("WARNING: h5py not found -- run Section 0 first")

try:
    import minari
    HAS_MINARI = True
    print("minari loaded")
except ImportError:
    HAS_MINARI = False



# =============================================================================
# SECTION 2  --  CONFIGURATION
# =============================================================================
class Config:
    # Environment
    ENV_NAME        = "FrankaKitchen-v1"
    ENV_TASKS       = ["microwave", "kettle", "light switch", "slide cabinet"]
    STATE_DIM       = None          # resolved dynamically from the env
    ACTION_DIM      = 9             # 7 arm joints + 2 gripper
    MAX_EPISODE_LEN = 280

    # Dataset: kitchen-complete-v2 (prefer Minari; HDF5 fallback)
    DATASET_URLS = [
        "http://rail.eecs.berkeley.edu/datasets/offline_rl/kitchen/kitchen_complete-v0.hdf5",
    ]
    DATASET_PATH = "./kitchen_complete-v2.hdf5"
    MINARI_DATASET_ID = "D4RL/kitchen/complete-v2"  # kitchen-complete-v2 for training
    PREFER_MINARI = True  # use Minari so obs structure matches FrankaKitchen-v1

    # RST Data Augmentation
    RST_K       = 5
    RST_SEG_LEN = 50
    RST_N_AUG   = 500

    # FlowPolicy hyperparameters (Consistency Flow Matching)
    FLOW_EPOCHS       = 3000
    LR_FLOW           = 5e-5
    BATCH_SIZE        = 128
    FLOW_HIDDEN       = 512
    FLOW_T_EMB_DIM    = 256
    FLOW_NUM_SEGMENTS = 3
    FLOW_EPS          = 0.01
    FLOW_DELTA        = 0.02   # segment margin (t + delta)
    FLOW_DELTA_T      = 0.01   # inference step size (dt); smaller = more steps, better action quality
    FLOW_BOUNDARY     = 1
    FLOW_ALPHA        = 1e-5   # velocity consistency weight
    ACTION_HORIZON    = 2
    OBS_HORIZON       = 4
    FLOW_DEPTH        = 6
    FLOW_N_STEPS      = 100    # inference Euler steps (match 1/FLOW_DELTA_T for t in [0,1])

    # FQL
    FQL_HIDDEN = 512
    FQL_DEPTH  = 4
    FQL_GAMMA  = 0.99
    FQL_TAU    = 0.005
    FQL_ALPHA  = 8.0    # stronger distillation to flow prior
    FQL_BETA   = 0.2    # use flow action in TD target (stability blend)
    FQL_Q_CLIP = 150.0  # clamp target Q range
    FQL_ACTOR_DELAY = 2 # update actor every N critic steps
    FQL_BC_COEF = 0.7   # BC loss toward batch action (stronger imitation for task completion)

    REWARD_SCALE = 0.1  # reward scaling for critic

    # Training (FlowPolicy uses FLOW_EPOCHS, LR_FLOW, BATCH_SIZE above)
    LR_FQL      = 3e-4
    FQL_EPOCHS  = 200
    EVAL_EVERY  = 20
    N_EVAL_EPS  = 5

    # Misc
    SEED          = 3
    DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
    SAVE_PATH     = "./checkpoints"

cfg = Config()
os.makedirs(cfg.SAVE_PATH, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.SEED)
print(f"Device: {cfg.DEVICE}")


# =============================================================================
# SECTION 3  --  ENVIRONMENT WRAPPER
# =============================================================================
class FlatObsWrapper(gym.Wrapper):
    """
    Flatten dict/tuple/array observations into a single float32 vector.

    FrankaKitchen-v1 returns a Dict observation space where some sub-spaces
    report shape=None (e.g. achieved_goal / desired_goal).  Inspecting
    .spaces is therefore unreliable; instead we do a real reset() and
    measure the flattened array directly.
    """

    def _flatten(self, obs):
        """Recursively extract all leaf numeric values and concatenate."""
        def _collect(o):
            if isinstance(o, dict):
                parts = []
                for v in o.values():
                    parts.extend(_collect(v))
                return parts
            try:
                arr = np.asarray(o, dtype=np.float32).ravel()
                return [arr] if arr.size > 0 else []
            except (TypeError, ValueError):
                return []
        parts = _collect(obs)
        return np.concatenate(parts) if parts else np.zeros(1, dtype=np.float32)

    def __init__(self, env):
        super().__init__(env)

        # Probe the actual flat dim with a real reset (avoids None-shape issues)
        result   = env.reset()
        raw_obs  = result[0] if isinstance(result, tuple) else result
        probe    = self._flatten(raw_obs)
        flat_dim = probe.shape[0]

        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=(flat_dim,), dtype=np.float32)
        self._flat_dim = flat_dim
        print(f"[FlatObsWrapper] flat obs dim = {flat_dim}")

    def reset(self, **kwargs):
        result = self.env.reset(**kwargs)
        obs, info = result if isinstance(result, tuple) else (result, {})
        return self._flatten(obs), info

    def step(self, action):
        result = self.env.step(action)
        if len(result) == 5:                  # gymnasium API
            obs, rew, term, trunc, info = result
            done = bool(term or trunc)
        else:                                  # gym API
            obs, rew, done, info = result
        return self._flatten(obs), float(rew), done, info


def build_env():
    env = gym.make(
        cfg.ENV_NAME,
        tasks_to_complete=cfg.ENV_TASKS,
        terminate_on_tasks_completed=False,
        remove_task_when_completed=False,
    )
    return FlatObsWrapper(env)




def build_render_env():
    """Build env with rgb_array rendering for video capture."""
    env = gym.make(
        cfg.ENV_NAME,
        tasks_to_complete=cfg.ENV_TASKS,
        terminate_on_tasks_completed=False,
        remove_task_when_completed=False,
        render_mode="rgb_array",
    )
    return FlatObsWrapper(env)

def infer_state_dim() -> int:
    """Create a temporary env to read the flat obs dim."""
    env = build_env()
    dim = env.observation_space.shape[0]
    env.close()
    cfg.STATE_DIM = dim
    print(f"STATE_DIM = {dim}")
    return dim


# =============================================================================
# SECTION 4  --  DATASET LOADING  (no d4rl required)
# =============================================================================
def download_dataset():
    """Download the HDF5 dataset if not already cached.  Tries multiple URLs."""
    if os.path.exists(cfg.DATASET_PATH):
        size_mb = os.path.getsize(cfg.DATASET_PATH) / 1e6
        print(f"Found cached dataset: {cfg.DATASET_PATH}  ({size_mb:.1f} MB)")
        return True

    def _progress(n_blocks, block_size, total_size):
        done = min(n_blocks * block_size, total_size)
        pct  = done / total_size * 100 if total_size > 0 else 0
        print(f"\r  {done/1e6:.1f}/{total_size/1e6:.1f} MB  ({pct:.0f}%)",
              end="", flush=True)

    for url in cfg.DATASET_URLS:
        try:
            print(f"Downloading dataset from {url} ...")
            urllib.request.urlretrieve(url, cfg.DATASET_PATH, _progress)
            print(f"\nSaved to {cfg.DATASET_PATH}")
            return True
        except Exception as e:
            print(f"\n  Failed: {e}")
            if os.path.exists(cfg.DATASET_PATH):
                os.remove(cfg.DATASET_PATH)

    print("All direct download URLs failed.")
    return False


def _flatten_single_obs(obs):
    """Recursively flatten one observation (dict/list/array) to 1D float32 array."""
    if isinstance(obs, dict):
        parts = []
        for k in sorted(obs.keys()):
            parts.append(_flatten_single_obs(obs[k]))
        return np.concatenate(parts)
    if isinstance(obs, (list, tuple)):
        return np.concatenate([_flatten_single_obs(x) for x in obs])
    return np.asarray(obs, dtype=np.float32).ravel()


def _minari_obs_length(raw_obs):
    """Get number of timesteps from nested dict/array/list (Minari kitchen has dict of dicts)."""
    if isinstance(raw_obs, np.ndarray):
        return raw_obs.shape[0] if raw_obs.ndim >= 1 else 1
    if isinstance(raw_obs, dict):
        return _minari_obs_length(raw_obs[next(iter(raw_obs))])
    if isinstance(raw_obs, (list, tuple)):
        return len(raw_obs)
    return 1


def _minari_obs_at_t(raw_obs, t):
    """Extract single observation at time index t from nested structure."""
    if isinstance(raw_obs, np.ndarray):
        return raw_obs[t]
    if isinstance(raw_obs, dict):
        return {k: _minari_obs_at_t(v, t) for k, v in raw_obs.items()}
    if isinstance(raw_obs, (list, tuple)):
        return raw_obs[t]
    return raw_obs


def _minari_obs_to_array(raw_obs):
    """Convert Minari episode observations (any structure) to (T, D) float32 array."""
    if isinstance(raw_obs, np.ndarray):
        if raw_obs.dtype == object or (raw_obs.size > 0 and isinstance(raw_obs.flat[0], dict)):
            return np.array([_flatten_single_obs(o) for o in raw_obs], dtype=np.float32)
        return np.asarray(raw_obs, dtype=np.float32)
    if isinstance(raw_obs, dict):
        first = raw_obs[next(iter(raw_obs))]
        if isinstance(first, np.ndarray) and first.ndim >= 1:
            parts = []
            for k in sorted(raw_obs.keys()):
                v = raw_obs[k]
                if isinstance(v, dict):
                    v = _minari_obs_to_array(v)
                v = np.asarray(v, dtype=np.float32)
                if v.ndim == 1:
                    v = v.reshape(-1, 1)
                parts.append(v)
            return np.concatenate(parts, axis=-1)
        # Nested dict (e.g. kitchen: observation, achieved_goal, desired_goal) -> index by time
        T = _minari_obs_length(raw_obs)
        if T <= 0:
            return np.zeros((0, 0), dtype=np.float32)
        return np.array([_flatten_single_obs(_minari_obs_at_t(raw_obs, t)) for t in range(T)], dtype=np.float32)
    if isinstance(raw_obs, (list, tuple)):
        if len(raw_obs) == 0:
            return np.zeros((0, 0), dtype=np.float32)
        return np.array([_flatten_single_obs(o) for o in raw_obs], dtype=np.float32)
    return np.asarray(raw_obs, dtype=np.float32).reshape(1, -1)


def _load_via_minari() -> Dict:
    """Load kitchen-complete dataset via Minari (official d4rl successor)."""
    dataset_id = cfg.MINARI_DATASET_ID
    print(f"[Minari] Loading dataset '{dataset_id}' ...")

    local_datasets = minari.list_local_datasets()
    if dataset_id not in local_datasets:
        print(f"[Minari] Downloading '{dataset_id}' ...")
        minari.download_dataset(dataset_id)

    dataset = minari.load_dataset(dataset_id)

    obs_all, acts_all, rews_all, dones_all = [], [], [], []
    for ep in dataset.iterate_episodes():
        raw_obs = ep.observations
        acts = np.asarray(ep.actions, dtype=np.float32)
        rews = np.asarray(ep.rewards, dtype=np.float32)
        obs = _minari_obs_to_array(raw_obs)

        T = min(len(obs), len(acts))
        obs, acts, rews = obs[:T], acts[:T], rews[:T]
        terms = np.zeros(T, dtype=bool)
        terms[-1] = True
        obs_all.append(obs)
        acts_all.append(acts)
        rews_all.append(rews)
        dones_all.append(terms)

    # #region agent log
    try:
        import json, time
        _logpath = os.path.join(os.getcwd(), "debug.log")
        with open(_logpath, "a") as _f:
            _f.write(json.dumps({"message": "_load_via_minari after loop", "data": {"num_episodes": len(obs_all), "obs_shapes": [tuple(o.shape) for o in obs_all[:5]]}, "hypothesisId": "D", "timestamp": int(time.time()*1000)}) + "\n")
    except Exception:
        pass
    # #endregion
    D_max = max(o.shape[1] for o in obs_all)
    obs_padded = []
    for o in obs_all:
        T, D = o.shape
        if D < D_max:
            o = np.concatenate([o, np.zeros((T, D_max - D), dtype=np.float32)], axis=1)
        elif D > D_max:
            o = o[:, :D_max]
        obs_padded.append(o)
    obs   = np.concatenate(obs_padded).astype(np.float32)
    acts  = np.concatenate(acts_all).astype(np.float32)
    rews  = np.concatenate(rews_all).astype(np.float32)
    dones = np.concatenate(dones_all)

    print(f"[Minari] Loaded {len(obs)} transitions, "
          f"{dones.sum()} episode ends, obs_dim={obs.shape[1]}")
    return obs, acts, rews, dones


def load_kitchen_dataset() -> Dict:
    obs, acts, rewards, dones = None, None, None, None
    minari_loaded = False
    if getattr(cfg, "PREFER_MINARI", False) and HAS_MINARI:
        try:
            obs, acts, rewards, dones = _load_via_minari()
            minari_loaded = True
        except Exception as e:
            print(f"[Minari] Failed: {e}, falling back to HDF5")
    if not minari_loaded:
        hdf5_ok = download_dataset()
        if hdf5_ok:
            with h5py.File(cfg.DATASET_PATH, "r") as f:
                obs      = f["observations"][:].astype(np.float32)
                acts     = f["actions"][:].astype(np.float32)
                rewards  = f["rewards"][:].astype(np.float32)
                terms    = f["terminals"][:].astype(bool)
                timeouts = (f["timeouts"][:].astype(bool)
                            if "timeouts" in f else np.zeros(len(obs), bool))
                dones    = terms | timeouts
            print(f"Loaded {len(obs)} transitions, {dones.sum()} episodes ends, "
                  f"obs_dim={obs.shape[1]}")
        elif HAS_MINARI:
            print("Falling back to Minari ...")
            obs, acts, rewards, dones = _load_via_minari()
        else:
            raise RuntimeError(
                "Cannot download dataset: all URLs failed and minari is not "
                "installed.  Run:  pip install minari")

    if cfg.STATE_DIM is None:
        cfg.STATE_DIM = obs.shape[1]
        print(f"STATE_DIM inferred from dataset: {cfg.STATE_DIM}")

    D = obs.shape[1]
    if D > cfg.STATE_DIM:
        obs = obs[:, :cfg.STATE_DIM]
        D = cfg.STATE_DIM
    elif D < cfg.STATE_DIM:
        obs = np.concatenate([obs,
                               np.zeros((len(obs), cfg.STATE_DIM - D),
                                        dtype=np.float32)], axis=1)

    # Normalize: use stats only on first D dims so padded dims don't blow up at inference
    obs_mean = np.zeros((1, cfg.STATE_DIM), dtype=np.float32)
    obs_std  = np.ones((1, cfg.STATE_DIM), dtype=np.float32)
    obs_mean[0, :D] = obs[:, :D].mean(0)
    obs_std[0, :D]  = obs[:, :D].std(0) + 1e-6
    obs_n = (obs - obs_mean) / obs_std

    act_min = acts.min(0, keepdims=True)
    act_max = acts.max(0, keepdims=True)
    acts_n  = 2.0 * (acts - act_min) / (act_max - act_min + 1e-6) - 1.0

    rewards = rewards * cfg.REWARD_SCALE

    episodes, t0 = [], 0
    # #region agent log
    skipped_len1 = 0
    # #endregion
    for t in range(len(obs_n)):
        if dones[t] or t == len(obs_n) - 1:
            ep = {"obs":  obs_n[t0:t+1],
                  "acts": acts_n[t0:t+1],
                  "rews": rewards[t0:t+1]}
            if len(ep["obs"]) >= 1:
                episodes.append(ep)
            else:
                skipped_len1 += 1
            t0 = t + 1

    # #region agent log
    try:
        import json, time
        _logpath = os.path.join(os.getcwd(), "debug.log")
        with open(_logpath, "a") as _f:
            _f.write(json.dumps({"message": "load_kitchen_dataset after parse", "data": {"len_episodes": len(episodes), "len_obs_n": len(obs_n), "dones_sum": int(dones.sum()), "skipped_len1": skipped_len1}, "hypothesisId": "A,B,E", "timestamp": int(time.time()*1000)}) + "\n")
    except Exception:
        pass
    # #endregion
    avg_len = np.mean([len(e["obs"]) for e in episodes]) if episodes else 0
    print(f"Parsed {len(episodes)} episodes  (avg len {avg_len:.1f})")

    return {"episodes": episodes,
            "obs_mean": obs_mean, "obs_std": obs_std,
            "act_min":  act_min,  "act_max": act_max,
            "data_obs_dim": D}


# =============================================================================
# SECTION 5  --  RST  (Retrieval-based Sub-Trajectory Augmentation)
# =============================================================================
class RST:
    """
    Augments a small offline dataset by stitching sub-trajectories
    from different episodes via nearest-neighbour retrieval on states.
    (RST / retrieval-based sub-trajectory augmentation.)

    For each synthetic trajectory:
      1. Pick a seed state from a random anchor episode.
      2. Retrieve K nearest-neighbour (ep, step) pairs by L2 state distance
         (excluding the anchor episode to ensure novelty).
      3. Append the matching sub-trajectory segment; update the current state.
      4. Repeat until MAX_EPISODE_LEN is reached.
    """

    def __init__(self, episodes: List[Dict], cfg: Config, data_obs_dim: Optional[int] = None):
        self.eps = episodes
        self.cfg = cfg
        # Use only first data_obs_dim for KNN when obs are padded (e.g. 60 vs 81)
        self.data_obs_dim = data_obs_dim
        self._build_index()

    def _build_index(self):
        states, refs = [], []
        for eid, ep in enumerate(self.eps):
            for t in range(len(ep["obs"])):
                states.append(ep["obs"][t])
                refs.append((eid, t))
        self.index = np.array(states, dtype=np.float32)   # (N, D)
        self.refs  = refs
        d = self.data_obs_dim if self.data_obs_dim is not None else self.index.shape[1]
        print(f"[RST] Indexed {len(states)} states from {len(self.eps)} episodes (KNN on first {d} dims)")

    def _knn(self, query: np.ndarray, k: int, excl: int = -1):
        dim = self.data_obs_dim if self.data_obs_dim is not None else self.index.shape[1]
        idx = self.index[:, :dim]
        q = query[:dim] if query.shape[0] > dim else query
        if q.shape[0] < dim:
            q = np.pad(q, (0, dim - q.shape[0]), mode="constant", constant_values=0.0)
        dists = ((idx - q[None]) ** 2).sum(-1)
        for i, (eid, _) in enumerate(self.refs):
            if eid == excl:
                dists[i] = np.inf
        finite = int((dists < np.inf).sum())
        if finite == 0:
            return []
        kk   = min(k, finite)
        top  = np.argpartition(dists, kk - 1)[:kk]
        top  = top[np.argsort(dists[top])]
        return [self.refs[i] for i in top]

    def _stitch_one(self) -> Optional[Dict]:
        if len(self.eps) == 0:
            return None
        seg = self.cfg.RST_SEG_LEN
        K   = self.cfg.RST_K
        aeid = random.randrange(len(self.eps))
        aep  = self.eps[aeid]
        if len(aep["obs"]) < seg:
            return None
        t0    = random.randrange(max(1, len(aep["obs"]) - seg))
        state = aep["obs"][t0].copy()
        os_l, ac_l, rw_l = [], [], []
        for _ in range(self.cfg.MAX_EPISODE_LEN // seg):
            nbrs = self._knn(state, K, excl=aeid)
            if not nbrs:
                break
            eid, t = random.choice(nbrs)
            ep     = self.eps[eid]
            te     = min(t + seg, len(ep["obs"]))
            if te <= t:
                break
            os_l.append(ep["obs"][t:te])
            ac_l.append(ep["acts"][t:te])
            rw_l.append(ep["rews"][t:te])
            state = ep["obs"][te - 1].copy()
            if sum(len(x) for x in os_l) >= self.cfg.MAX_EPISODE_LEN:
                break
        if not os_l:
            return None
        return {"obs":  np.concatenate(os_l),
                "acts": np.concatenate(ac_l),
                "rews": np.concatenate(rw_l)}

    def augment(self, n_aug: int) -> List[Dict]:
        if len(self.eps) == 0:
            print("[RST] No episodes to augment (len(eps)=0), skipping.")
            return []
        out, tries = [], 0
        pbar = tqdm(total=n_aug, desc="[RST] Augmenting")
        while len(out) < n_aug and tries < n_aug * 15:
            ep = self._stitch_one()
            if ep is not None:
                out.append(ep)
                pbar.update(1)
            tries += 1
        pbar.close()
        total = sum(len(e["obs"]) for e in out)
        print(f"[RST] {len(out)} augmented episodes ({total} steps)")
        return out


# =============================================================================
# SECTION 6  --  PYTORCH DATASET
# =============================================================================
class TransitionDataset(Dataset):
    def __init__(self, episodes: List[Dict]):
        os_l, ac_l, rw_l, no_l, dn_l = [], [], [], [], []
        for ep in episodes:
            T = len(ep["obs"])
            for t in range(T - 1):
                os_l.append(ep["obs"][t]);    ac_l.append(ep["acts"][t])
                rw_l.append(ep["rews"][t]);   no_l.append(ep["obs"][t+1])
                dn_l.append(0.0)
            if T > 0:
                os_l.append(ep["obs"][-1]);   ac_l.append(ep["acts"][-1])
                rw_l.append(ep["rews"][-1]);  no_l.append(ep["obs"][-1])
                dn_l.append(1.0)
        mk = lambda lst, t=torch.float32: torch.tensor(np.array(lst), dtype=t)
        self.obs      = mk(os_l)
        self.acts     = mk(ac_l)
        self.rewards  = mk(rw_l)
        self.next_obs = mk(no_l)
        self.dones    = mk(dn_l)
        print(f"TransitionDataset: {len(self.obs)} transitions")

    def __len__(self):  return len(self.obs)
    def __getitem__(self, i):
        return (self.obs[i], self.acts[i], self.rewards[i],
                self.next_obs[i], self.dones[i])


# =============================================================================
# SECTION 7  --  BUILDING BLOCKS + FlowPolicy (Consistency Flow Matching)
#    From FlowPolicy/flow_policy_3d: sde_lib.py, losses.py, policy logic.
# =============================================================================
def sinusoidal_emb(t: torch.Tensor, dim: int) -> torch.Tensor:
    half  = dim // 2
    freqs = torch.exp(-math.log(10000) *
                      torch.arange(half, device=t.device, dtype=torch.float32)
                      / max(half - 1, 1))
    args  = t.float()[:, None] * freqs[None]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class MLP(nn.Module):
    def __init__(self, in_d, out_d, hid, depth, ln=True):
        super().__init__()
        layers, d = [], in_d
        for _ in range(depth - 1):
            layers += [nn.Linear(d, hid)]
            if ln: layers += [nn.LayerNorm(hid)]
            layers += [nn.SiLU()]
            d = hid
        layers += [nn.Linear(d, out_d)]
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)


# ----- ConsistencyFM (from FlowPolicy/flow_policy_3d/sde_lib.py) -----
class ConsistencyFM:
    """Consistency Flow Matching SDE. Used for training and inference."""
    def __init__(self, init_type='gaussian', noise_scale=1.0, sigma_var=0.0, sample_N=10):
        self.sample_N = sample_N
        self.init_type = init_type
        self.noise_scale = noise_scale
        self.sigma_t = lambda t: (1.0 - t) * sigma_var
        self.consistencyfm_hyperparameters = {
            'delta': 1e-3,
            'num_segments': 2,
            'boundary': 1,
            'alpha': 1e-5,
        }

    def T(self):
        return 1.0

    def get_z0(self, batch):
        B, *rest = batch.shape
        if self.init_type == 'gaussian':
            return torch.randn(B, *rest, device=batch.device, dtype=batch.dtype) * self.noise_scale
        raise NotImplementedError(self.init_type)


# ----- Consistency Flow Matching Loss (from FlowPolicy/flow_policy_3d/losses.py) -----
def consistency_flow_matching_loss(model_fn, batch, sde_hyperparams, reduce_mean=True):
    """batch: (B, 1, action_dim) or (B, action_dim); model_fn(xt, t) -> velocity same shape."""
    if batch.dim() == 2:
        batch = batch.unsqueeze(1)  # (B, action_dim) -> (B, 1, action_dim)
    B, H, D = batch.shape
    z0 = torch.randn(B, H, D, device=batch.device, dtype=batch.dtype)
    eps = sde_hyperparams.get('eps', 1e-3)
    t = torch.rand(B, device=batch.device) * (sde_hyperparams.get('T', 1.0) - eps) + eps
    r = torch.clamp(t + sde_hyperparams['delta'], max=1.0)
    t_expand = t.view(-1, 1, 1).expand(B, H, D)
    r_expand = r.view(-1, 1, 1).expand(B, H, D)
    xt = t_expand * batch + (1.0 - t_expand) * z0
    xr = r_expand * batch + (1.0 - r_expand) * z0
    num_segments = sde_hyperparams['num_segments']
    boundary = sde_hyperparams['boundary']
    delta = sde_hyperparams['delta']
    alpha = sde_hyperparams['alpha']
    segments = torch.linspace(0, 1, num_segments + 1, device=batch.device)
    seg_indices = torch.searchsorted(segments, t, side='left').clamp(min=1)
    segment_ends = segments[seg_indices]
    segment_ends_expand = segment_ends.view(-1, 1, 1).expand(B, H, D)
    x_at_segment_ends = segment_ends_expand * batch + (1.0 - segment_ends_expand) * z0

    def f_euler(te, se, x, v):
        return x + (se - te) * v

    def threshold_based_f_euler(te, se, x, v, thresh, x_end):
        if isinstance(thresh, int) and thresh == 0:
            return x_end
        less = te < thresh
        return torch.where(less, f_euler(te, se, x, v), x_end)

    vt = model_fn(xt, t)
    with torch.no_grad():
        vr = model_fn(xr, r) if not (isinstance(boundary, int) and boundary == 0) else None
    if vr is not None:
        vr = torch.nan_to_num(vr)
    # Conditional flow matching: for linear path x_t = t*batch + (1-t)*z0, target velocity = batch - z0
    v_target = batch - z0
    losses_cfm = (vt - v_target).pow(2).reshape(B, -1).sum(-1).mean()
    ft = f_euler(t_expand, segment_ends_expand, xt, vt)
    fr = threshold_based_f_euler(r_expand, segment_ends_expand, xr, vr, boundary, x_at_segment_ends) if vr is not None else x_at_segment_ends
    losses_f = (ft - fr).pow(2).reshape(B, -1).mean(-1)
    if isinstance(boundary, int) and boundary == 0:
        losses_v = 0.0
    else:
        less_than_threshold = t_expand < boundary
        far_from_ends = (segment_ends - t).view(-1, 1, 1).expand(B, H, D) > 1.01 * delta
        losses_v = ((vt - vr).pow(2) * less_than_threshold.float() * far_from_ends.float()).reshape(B, -1).mean(-1)
    reduce_op = torch.mean if reduce_mean else (lambda x: 0.5 * x.sum())
    lambda_cfm = sde_hyperparams.get('lambda_cfm', 1.0)
    return reduce_op(losses_cfm * lambda_cfm + losses_f + alpha * (losses_v if isinstance(losses_v, torch.Tensor) else torch.zeros_like(losses_f)))


# ----- FlowPolicy: state-conditioned velocity + Consistency FM (flow_policy_3d logic) -----
class VelocityNet(nn.Module):
    """Velocity field v_theta(s, x_t, t). Used for both rectified flow and Consistency FM."""
    def __init__(self, state_dim, action_dim, cfg):
        super().__init__()
        H, T = cfg.FLOW_HIDDEN, cfg.FLOW_T_EMB_DIM
        self.t_dim = T
        self.inp_proj = nn.Linear(state_dim + action_dim, H)
        self.t_proj = nn.Linear(T, H)
        self.blocks = nn.ModuleList([
            nn.Sequential(nn.Linear(H, H), nn.LayerNorm(H), nn.SiLU())
            for _ in range(cfg.FLOW_DEPTH - 1)
        ])
        self.out = nn.Linear(H, action_dim)

    def forward(self, s, x_t, t):
        x_flat = x_t.reshape(x_t.shape[0], -1)
        h = self.inp_proj(torch.cat([s, x_flat], -1))
        h = h + self.t_proj(sinusoidal_emb(t, self.t_dim))
        for blk in self.blocks:
            h = h + blk(h)
        return self.out(h)


class FlowPolicy(nn.Module):
    """FlowPolicy with Consistency Flow Matching (FlowPolicy/flow_policy_3d)."""
    def __init__(self, state_dim, action_dim, cfg):
        super().__init__()
        self.vel = VelocityNet(state_dim, action_dim, cfg)
        self.adim = action_dim
        self.steps = getattr(cfg, 'FLOW_N_STEPS', 10)
        self.dt = getattr(cfg, 'FLOW_DELTA_T', 0.02)
        self.sde = ConsistencyFM('gaussian', noise_scale=1.0, sigma_var=0.0, sample_N=self.steps)
        self.cfm_hyper = {
            'delta': getattr(cfg, 'FLOW_DELTA', 0.02),
            'num_segments': getattr(cfg, 'FLOW_NUM_SEGMENTS', 3),
            'boundary': getattr(cfg, 'FLOW_BOUNDARY', 1),
            'alpha': getattr(cfg, 'FLOW_ALPHA', 1e-5),
            'eps': getattr(cfg, 'FLOW_EPS', 0.01),
            'T': self.sde.T(),
            'lambda_terminal': 0.05,
        }

    def _model_fn(self, s, xt, t):
        """xt (B, 1, action_dim) -> v (B, 1, action_dim)."""
        v = self.vel(s, xt, t)
        return v.unsqueeze(1) if v.dim() == 2 else v

    def flow_loss(self, s, a):
        batch = a.unsqueeze(1)
        def model_fn(xt, t):
            return self._model_fn(s, xt, t)
        loss_cfm = consistency_flow_matching_loss(model_fn, batch, self.cfm_hyper, reduce_mean=True)
        # Terminal regularizer: at t≈1 (at target action), velocity should be small to avoid overshoot
        t_term = torch.full((s.shape[0],), 0.99, device=s.device, dtype=s.dtype)
        v_at_target = self.vel(s, batch.squeeze(1), t_term)
        loss_terminal = (v_at_target.pow(2).mean()) * self.cfm_hyper.get('lambda_terminal', 0.1)
        return loss_cfm + loss_terminal

    @torch.no_grad()
    def sample(self, s, n_steps=None):
        dt = self.dt
        n = n_steps or (int(round(1.0 / dt)) if dt else self.steps)
        if not dt:
            dt = 1.0 / self.steps
        x = torch.randn(s.shape[0], self.adim, device=s.device)
        for i in range(n):
            num_t = (i + 0.5) * dt  # midpoint in [0, 1]
            num_t = min(max(num_t, 1e-5), 1.0 - 1e-5)
            t = torch.full((s.shape[0],), num_t, device=s.device)
            x = x + self.vel(s, x, t) * dt
        return x.clamp(-1.0, 1.0)

    def forward(self, s):
        return self.sample(s)


# =============================================================================
# SECTION 8  --  FQL  (exactly as in fql/agents/fql.py: bc_flow + distill + q)
# =============================================================================
class TwinQ(nn.Module):
    def __init__(self, state_dim, action_dim, cfg):
        super().__init__()
        d = state_dim + action_dim
        self.q1 = MLP(d, 1, cfg.FQL_HIDDEN, cfg.FQL_DEPTH)
        self.q2 = MLP(d, 1, cfg.FQL_HIDDEN, cfg.FQL_DEPTH)

    def both(self, s, a):
        x = torch.cat([s, a], -1)
        return self.q1(x).squeeze(-1), self.q2(x).squeeze(-1)

    def min(self, s, a):
        q1, q2 = self.both(s, a)
        return torch.min(q1, q2)


class OnestepActor(nn.Module):
    """One-step flow policy f_phi(s, z) -- as in fql/agents/fql.py actor_onestep_flow."""
    def __init__(self, state_dim, action_dim, cfg):
        super().__init__()
        self.net  = MLP(state_dim + action_dim, action_dim,
                        cfg.FQL_HIDDEN, cfg.FQL_DEPTH)
        self.adim = action_dim

    def forward(self, s, z=None):
        if z is None:
            z = torch.randn(s.shape[0], self.adim, device=s.device)
        return self.net(torch.cat([s, z], -1)).tanh()


class FQL(nn.Module):
    """
    Flow Q-Learning (FQL) exactly as in fql/agents/fql.py:
    - Critic: TD loss, next_actions from policy (one-step), target_critic.
    - Actor: bc_flow_loss (velocity matching) + alpha * distill_loss (one-step vs flow) + q_loss.
    - compute_flow_actions: Euler from noise using flow (BC flow).
    """
    def __init__(self, state_dim, action_dim, flow: FlowPolicy, cfg):
        super().__init__()
        self.flow   = flow
        self.actor  = OnestepActor(state_dim, action_dim, cfg)
        self.critic = TwinQ(state_dim, action_dim, cfg)
        self.target = copy.deepcopy(self.critic)
        for p in self.target.parameters():
            p.requires_grad_(False)
        self.gamma   = cfg.FQL_GAMMA
        self.tau     = cfg.FQL_TAU
        self.alpha   = cfg.FQL_ALPHA
        self.beta    = cfg.FQL_BETA
        self.q_clip  = cfg.FQL_Q_CLIP
        self.adim    = action_dim
        self.flow_steps = getattr(cfg, 'FLOW_N_STEPS', 10)

    @torch.no_grad()
    def soft_update(self):
        for p, pt in zip(self.critic.parameters(), self.target.parameters()):
            pt.data.lerp_(p.data, self.tau)

    def critic_loss(self, s, a, r, s2, d):
        with torch.no_grad():
            next_actions = self.actor(s2)
            next_actions = next_actions.clamp(-1.0, 1.0)
            next_q = self.target.min(s2, next_actions)
            q_t = r + self.gamma * (1.0 - d) * next_q
            q_t = q_t.clamp(-self.q_clip, self.q_clip)
        q1, q2 = self.critic.both(s, a)
        return F.smooth_l1_loss(q1, q_t) + F.smooth_l1_loss(q2, q_t)

    def compute_flow_actions(self, observations, noises):
        """Compute actions from BC flow using Euler (fql.py compute_flow_actions)."""
        actions = noises
        for i in range(self.flow_steps):
            t = torch.full((observations.shape[0],), i / self.flow_steps, device=observations.device)
            vels = self.flow.vel(observations, actions, t)
            actions = actions + vels / self.flow_steps
        return actions.clamp(-1.0, 1.0)

    def actor_loss(self, s, a_batch):
        """FQL actor loss: bc_flow_loss + alpha * distill_loss + q_loss (fql.py)."""
        B = s.shape[0]
        # 1) BC flow loss: velocity matching on (s, a_batch)
        x_0 = torch.randn(B, self.adim, device=s.device)
        x_1 = a_batch
        t = torch.rand(B, device=s.device).unsqueeze(1)
        x_t = (1.0 - t) * x_0 + t * x_1
        vel = x_1 - x_0
        pred_vel = self.flow.vel(s, x_t, t.squeeze(-1))
        bc_flow_loss = F.mse_loss(pred_vel, vel)
        # 2) Distillation: one-step vs flow (Euler)
        noises = torch.randn(B, self.adim, device=s.device)
        target_flow_actions = self.compute_flow_actions(s, noises)
        actor_actions = self.actor(s, noises)
        distill_loss = F.mse_loss(actor_actions, target_flow_actions)
        # 3) Q loss
        actor_actions = actor_actions.clamp(-1.0, 1.0)
        q = self.critic.min(s, actor_actions)
        q_loss = -q.mean()
        loss = bc_flow_loss + self.alpha * distill_loss + q_loss
        return loss, q.mean().detach(), distill_loss.detach(), bc_flow_loss.detach()

    @torch.no_grad()
    def act(self, s):
        return self.actor(s)


# =============================================================================
# SECTION 10  --  EVALUATION
# =============================================================================
def run_eval(env, policy_fn, obs_mean, obs_std,
             act_min, act_max, n_eps, device) -> Dict:
    rewards, tasks = [], []
    for _ in range(n_eps):
        result = env.reset()
        obs    = result[0] if isinstance(result, tuple) else result
        done, ep_r, ep_t = False, 0.0, set()
        while not done:
            # Normalise + align obs dim
            obs_n = (obs - obs_mean.squeeze()) / obs_std.squeeze()
            if len(obs_n) < cfg.STATE_DIM:
                obs_n = np.pad(obs_n, (0, cfg.STATE_DIM - len(obs_n)))
            else:
                obs_n = obs_n[:cfg.STATE_DIM]
            s_t  = torch.tensor(obs_n, dtype=torch.float32,
                                device=device).unsqueeze(0)
            a_t  = policy_fn(s_t).cpu().numpy()[0]
            # De-normalise action
            a_np = (a_t + 1.0) / 2.0 * (act_max - act_min).squeeze() \
                   + act_min.squeeze()
            a_np = np.clip(a_np, env.action_space.low, env.action_space.high)
            step_result = env.step(a_np)
            if len(step_result) == 5:
                obs, r, term, trunc, info = step_result
                done = bool(term or trunc)
            else:
                obs, r, done, info = step_result
            ep_r += r
            # FrankaKitchen: completed tasks (key may vary by gymnasium-robotics version)
            completed = info.get("completed_tasks", info.get("completed_task", []))
            if isinstance(completed, (list, tuple)):
                for tk in completed:
                    ep_t.add(tk)
            elif isinstance(completed, (int, float)):
                ep_t.add(completed)
        rewards.append(ep_r)
        tasks.append(len(ep_t))
    return {"mean_reward": float(np.mean(rewards)),
            "std_reward":  float(np.std(rewards)),
            "mean_tasks":  float(np.mean(tasks))}


# =============================================================================
# SECTION 11  --  TRAINING
# =============================================================================
def train():
    print("=" * 60)
    print("  FlowPolicy + FQL + RST  |  Franka Kitchen")
    print("=" * 60)

    # Resolve STATE_DIM from the live environment
    infer_state_dim()
    S, A = cfg.STATE_DIM, cfg.ACTION_DIM

    # Load + augment dataset
    data    = load_kitchen_dataset()
    eps_raw = data["episodes"]
    if len(eps_raw) == 0:
        raise ValueError("No episodes in dataset. Check that dones/terminals mark episode ends and that the dataset has multiple steps per episode.")
    # #region agent log
    try:
        import json, time
        _logpath = os.path.join(os.getcwd(), "debug.log")
        with open(_logpath, "a") as _f:
            _f.write(json.dumps({"message": "train before RST", "data": {"len_eps_raw": len(eps_raw), "episodes_in_data": "episodes" in data}, "hypothesisId": "A,C", "timestamp": int(time.time()*1000)}) + "\n")
    except Exception:
        pass
    # #endregion
    rst     = RST(eps_raw, cfg, data.get("data_obs_dim"))
    # #region agent log
    try:
        _logpath = os.path.join(os.getcwd(), "debug.log")
        with open(_logpath, "a") as _f:
            _f.write(json.dumps({"message": "RST.augment entry", "data": {"len_self_eps": len(rst.eps)}, "hypothesisId": "A", "timestamp": int(time.time()*1000)}) + "\n")
    except Exception:
        pass
    # #endregion
    eps_aug = rst.augment(cfg.RST_N_AUG)
    all_eps = eps_raw + eps_aug
    print(f"Data: {len(eps_raw)} original + {len(eps_aug)} RST = {len(all_eps)} eps")

    # DataLoader
    dset   = TransitionDataset(all_eps)
    if len(dset) < 100:
        print("[WARNING] Very few transitions ({}). FlowPolicy may get NaN. Consider PREFER_MINARI=False to use HDF5.".format(len(dset)))
    loader = DataLoader(dset, batch_size=cfg.BATCH_SIZE,
                        shuffle=True, drop_last=True,
                        num_workers=2, pin_memory=(cfg.DEVICE == "cuda"))

    # Models
    flow = FlowPolicy(S, A, cfg).to(cfg.DEVICE)
    fql  = FQL(S, A, flow, cfg).to(cfg.DEVICE)

    flow_opt = Adam(flow.parameters(), lr=cfg.LR_FLOW / max(A, 1), weight_decay=1e-5)
    q_opt    = Adam(fql.critic.parameters(), lr=cfg.LR_FQL)
    a_opt    = Adam(fql.actor.parameters(),  lr=cfg.LR_FQL)

    flow_sch = CosineAnnealingLR(flow_opt, cfg.FLOW_EPOCHS)
    q_sch    = CosineAnnealingLR(q_opt,    cfg.FQL_EPOCHS)
    a_sch    = CosineAnnealingLR(a_opt,    cfg.FQL_EPOCHS)

    eval_env = build_env()
    eval_env.reset(seed=cfg.SEED + 100)

    # ==========================================================================
    # PHASE 1 -- FlowPolicy pre-training (Rectified Flow BC)
    # ==========================================================================
    print("\n[Phase 1] Pre-training FlowPolicy ...")
    best_r1 = -np.inf

    for epoch in range(1, cfg.FLOW_EPOCHS + 1):
        flow.train()
        losses = []
        for s, a, *_ in loader:
            s, a = s.to(cfg.DEVICE), a.to(cfg.DEVICE)
            loss = flow.flow_loss(s, a)
            flow_opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(flow.parameters(), 1.0)
            flow_opt.step()
            losses.append(loss.item())
        flow_sch.step()

        if epoch % cfg.EVAL_EVERY == 0 or epoch == 1:
            flow.eval()
            m = run_eval(eval_env, flow,
                         data["obs_mean"], data["obs_std"],
                         data["act_min"],  data["act_max"],
                         cfg.N_EVAL_EPS,   cfg.DEVICE)
            print(f"  [Flow {epoch:4d}]  loss={float(np.nanmean(losses)):.4f}  "
                  f"R={m['mean_reward']:.1f}+-{m['std_reward']:.1f}  "
                  f"tasks={m['mean_tasks']:.2f}")
            if m["mean_reward"] > best_r1:
                best_r1 = m["mean_reward"]
                torch.save(flow.state_dict(),
                           f"{cfg.SAVE_PATH}/flow_best.pt")

    print(f"[Phase 1] Best reward = {best_r1:.1f}")
    flow.load_state_dict(torch.load(f"{cfg.SAVE_PATH}/flow_best.pt",
                                    map_location=cfg.DEVICE))

    # ==========================================================================
    # PHASE 2 -- FQL Refinement (fql.py: bc_flow + distill + q; flow.vel trained with bc_flow_loss)
    # ==========================================================================
    print("\n[Phase 2] FQL Refinement ...")
    # Do NOT freeze flow: FQL trains flow.vel with bc_flow_loss (velocity matching)
    a_opt = Adam(list(fql.actor.parameters()) + list(flow.vel.parameters()), lr=cfg.LR_FQL)
    a_sch = CosineAnnealingLR(a_opt, cfg.FQL_EPOCHS)

    best_r2 = -np.inf

    global_step = 0
    for epoch in range(1, cfg.FQL_EPOCHS + 1):
        fql.train()
        q_ls, a_ls, q_vs = [], [], []

        for s, a, r, s2, d in loader:
            global_step += 1
            s, a  = s.to(cfg.DEVICE), a.to(cfg.DEVICE)
            r, s2 = r.to(cfg.DEVICE), s2.to(cfg.DEVICE)
            d     = d.to(cfg.DEVICE)

            # Critic
            cl = fql.critic_loss(s, a, r, s2, d)
            q_opt.zero_grad()
            cl.backward()
            nn.utils.clip_grad_norm_(fql.critic.parameters(), 1.0)
            q_opt.step()
            q_ls.append(cl.item())

            # Actor (delayed update for stability)
            if global_step % cfg.FQL_ACTOR_DELAY == 0:
                al, qv, _, _ = fql.actor_loss(s, a)
                a_opt.zero_grad()
                al.backward()
                nn.utils.clip_grad_norm_(fql.actor.parameters(), 1.0)
                a_opt.step()
                a_ls.append(al.item())
                q_vs.append(qv.item())

                fql.soft_update()

        q_sch.step()
        a_sch.step()

        if epoch % cfg.EVAL_EVERY == 0 or epoch == 1:
            fql.eval()
            m = run_eval(eval_env, fql.act,
                         data["obs_mean"], data["obs_std"],
                         data["act_min"],  data["act_max"],
                         cfg.N_EVAL_EPS,   cfg.DEVICE)
            print(f"  [FQL  {epoch:4d}]  "
                  f"q_loss={np.mean(q_ls):.4f}  "
                  f"a_loss={np.mean(a_ls) if a_ls else 0.0:.4f}  "
                  f"Q={np.mean(q_vs) if q_vs else 0.0:.2f}  "
                  f"R={m['mean_reward']:.1f}+-{m['std_reward']:.1f}  "
                  f"tasks={m['mean_tasks']:.2f}")
            if m["mean_reward"] > best_r2:
                best_r2 = m["mean_reward"]
                torch.save({"flow":   flow.state_dict(),
                            "actor":  fql.actor.state_dict(),
                            "critic": fql.critic.state_dict()},
                           f"{cfg.SAVE_PATH}/fql_best.pt")

    print(f"\n[Phase 2] Best FQL reward = {best_r2:.1f}")
    eval_env.close()
    return flow, fql, data


# =============================================================================
# SECTION 12  --  INFERENCE
# =============================================================================
def _load_policy_for_inference(ckpt_path: str):
    infer_state_dim()
    ckpt = torch.load(ckpt_path, map_location=cfg.DEVICE)
    flow = FlowPolicy(cfg.STATE_DIM, cfg.ACTION_DIM, cfg).to(cfg.DEVICE)
    fql  = FQL(cfg.STATE_DIM, cfg.ACTION_DIM, flow, cfg).to(cfg.DEVICE)
    flow.load_state_dict(ckpt["flow"])
    fql.actor.load_state_dict(ckpt["actor"])
    fql.eval()
    data = load_kitchen_dataset()
    return fql, data


def load_and_eval(ckpt_path: str, n_eps: int = 5):
    """Load checkpoint and run evaluation rollouts."""
    fql, data = _load_policy_for_inference(ckpt_path)
    env = build_env()
    m = run_eval(env, fql.act,
                 data["obs_mean"], data["obs_std"],
                 data["act_min"], data["act_max"],
                 n_eps, cfg.DEVICE)
    env.close()
    print(f"[Eval] R={m['mean_reward']:.2f}+-{m['std_reward']:.2f}  "
          f"tasks={m['mean_tasks']:.2f}")
    return m


def eval_and_save_video(ckpt_path: str, video_path: str = "franka_eval.mp4",
                        max_steps: int = 280, seed: int = 123, fps: int = 20):
    """Run one rollout on FrankaKitchen and save an MP4 video (Colab-friendly)."""
    fql, data = _load_policy_for_inference(ckpt_path)
    env = build_render_env()
    result = env.reset(seed=seed)
    obs = result[0] if isinstance(result, tuple) else result

    frames, ep_r, ep_tasks = [], 0.0, set()
    for _ in range(max_steps):
        frame = env.render()
        if frame is not None:
            frames.append(frame)

        obs_n = (obs - data["obs_mean"].squeeze()) / data["obs_std"].squeeze()
        if len(obs_n) < cfg.STATE_DIM:
            obs_n = np.pad(obs_n, (0, cfg.STATE_DIM - len(obs_n)))
        else:
            obs_n = obs_n[:cfg.STATE_DIM]

        s_t = torch.tensor(obs_n, dtype=torch.float32, device=cfg.DEVICE).unsqueeze(0)
        with torch.no_grad():
            a_t = fql.act(s_t).cpu().numpy()[0]

        a_np = (a_t + 1.0) / 2.0 * (data["act_max"] - data["act_min"]).squeeze() + data["act_min"].squeeze()
        a_np = np.clip(a_np, env.action_space.low, env.action_space.high)

        step_result = env.step(a_np)
        if len(step_result) == 5:
            obs, r, term, trunc, info = step_result
            done = bool(term or trunc)
        else:
            obs, r, done, info = step_result
        ep_r += float(r)
        completed = info.get("completed_tasks", info.get("completed_task", []))
        if isinstance(completed, (list, tuple)):
            for tk in completed:
                ep_tasks.add(tk)
        elif isinstance(completed, (int, float)):
            ep_tasks.add(completed)
        if done:
            break

    env.close()

    if len(frames) == 0:
        raise RuntimeError("No frames captured. Check render_mode and environment setup.")

    imageio.mimsave(video_path, frames, fps=fps)
    print(f"[Video Saved] {video_path}  frames={len(frames)}  R={ep_r:.2f}  tasks={len(ep_tasks)}")
    return {"video_path": video_path, "reward": ep_r, "tasks": len(ep_tasks), "frames": len(frames)}


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    flow, fql, data = train()
    print("Checkpoints saved to:", cfg.SAVE_PATH)


dm_control loaded
gymnasium 1.2.3 + gymnasium-robotics loaded
h5py loaded
minari loaded
Device: cuda
  FlowPolicy + FQL + RST  |  Franka Kitchen
[FlatObsWrapper] flat obs dim = 81
STATE_DIM = 81
[Minari] Loading dataset 'D4RL/kitchen/complete-v2' ...
[Minari] Downloading 'D4RL/kitchen/complete-v2' ...


namespace_metadata.json: 0.00B [00:00, ?B/s]

metadata.json: 0.00B [00:00, ?B/s]

namespace_metadata.json: 0.00B [00:00, ?B/s]

namespace_metadata.json: 0.00B [00:00, ?B/s]

...

namespace_metadata.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]


Dataset D4RL/kitchen/complete-v2 downloaded to /root/.minari/datasets/D4RL/kitchen/complete-v2
[Minari] Loaded 4209 transitions, 19 episode ends, obs_dim=81
Parsed 19 episodes  (avg len 221.5)
[RST] Indexed 4209 states from 19 episodes (KNN on first 81 dims)



[RST] Augmenting: 100%|██████████| 500/500 [00:02<00:00, 209.74it/s]


[RST] 500 augmented episodes (72293 steps)
Data: 19 original + 500 RST = 519 eps
TransitionDataset: 76502 transitions
[FlatObsWrapper] flat obs dim = 81

[Phase 1] Pre-training FlowPolicy ...
  [Flow    1]  loss=5.7774  R=0.0+-0.0  tasks=0.00
  [Flow   20]  loss=1.1887  R=0.0+-0.0  tasks=0.00
  [Flow   40]  loss=0.9892  R=0.0+-0.0  tasks=0.00
  [Flow   60]  loss=0.8940  R=0.0+-0.0  tasks=0.00
  [Flow   80]  loss=0.8203  R=0.0+-0.0  tasks=0.00
  [Flow  100]  loss=0.7682  R=0.0+-0.0  tasks=0.00
  [Flow  120]  loss=0.7332  R=0.0+-0.0  tasks=0.00
  [Flow  140]  loss=0.6921  R=0.0+-0.0  tasks=0.00
  [Flow  160]  loss=0.6498  R=0.0+-0.0  tasks=0.00
  [Flow  180]  loss=0.6347  R=0.0+-0.0  tasks=0.00
  [Flow  200]  loss=0.5929  R=0.0+-0.0  tasks=0.00
  [Flow  220]  loss=0.5806  R=0.0+-0.0  tasks=0.00
  [Flow  240]  loss=0.5502  R=0.0+-0.0  tasks=0.00
  [Flow  260]  loss=0.5423  R=0.0+-0.0  tasks=0.00
  [Flow  280]  loss=0.5220  R=0.0+-0.0  tasks=0.00
  [Flow  300]  loss=0.5009  R=0.0+-0.0  tas

In [ ]:
# Evaluate checkpoint
metrics = load_and_eval('./checkpoints/fql_best.pt', n_eps=5)
print(metrics)

# Save one rollout video
out = eval_and_save_video(
    ckpt_path='./checkpoints/fql_best.pt',
    video_path='franka_eval.mp4',
    max_steps=280,
    seed=123,
    fps=20,
)
print(out)

# Optional: display in Colab
from IPython.display import Video
Video('franka_eval.mp4', embed=True)